In [ ]:
import json
import os
import platform
import shutil
import subprocess
import sys
import textwrap
import zipfile
from datetime import datetime
from pathlib import Path

import pandas as pd
import torch

## 1. Проверка окружения и GPU

In [1]:
print("Python:", sys.version)
print("Platform:", platform.platform())
print("Текущая папка:", Path.cwd())
print("PyTorch:", torch.__version__)
print("CUDA build:", torch.version.cuda)
print("CUDA доступна:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError(
        "Kaggle GPU не обнаружена. Выберите GPU и перезапустите сессию."
    )

print("GPU:", torch.cuda.get_device_name(0))
print("GPU count:", torch.cuda.device_count())

os.system("nvidia-smi")

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Platform: Linux-6.12.90+-x86_64-with-glibc2.35
Текущая папка: /kaggle/working
PyTorch: 2.10.0+cu128
CUDA build: 12.8
CUDA доступна: True
GPU: Tesla T4
GPU count: 2
Sat Jul 25 18:55:32 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   5

0

## 2. Поиск и перенос проекта в `/kaggle/working`

Перед выполнением этой ячейки проект должен быть подключён к Notebook:

1. Откройте справа панель **Input**.
2. Нажмите **Add Input**.
3. Подключите Kaggle Dataset, в который загружен ZIP проекта.
4. После подключения перезапустите эту ячейку.

In [2]:
INPUT_ROOT = Path("/kaggle/input")
WORKING_ROOT = Path("/kaggle/working")
PROJECT_ROOT = WORKING_ROOT / "fl-attacks-lab"
STAGING_ROOT = WORKING_ROOT / "_fl_attacks_staging"

MANUAL_PROJECT_SOURCE: str | None = None


def print_input_contents(
        root: Path,
        max_depth: int = 4,
        max_items: int = 200,
) -> None:
    """Показывает, какие файлы Kaggle действительно подключил."""
    print(f"Содержимое {root}:")

    if not root.exists():
        print("  Каталог отсутствует")
        return

    displayed = 0

    for path in sorted(root.rglob("*")):
        try:
            relative = path.relative_to(root)
        except ValueError:
            continue

        if len(relative.parts) > max_depth:
            continue

        suffix = "/" if path.is_dir() else ""
        print(f"  {relative}{suffix}")
        displayed += 1

        if displayed >= max_items:
            print(f"  ... показаны первые {max_items} элементов")
            break

    if displayed == 0:
        print("  Каталог пуст — Dataset с проектом не подключён")


def is_project_root(path: Path) -> bool:
    """Проверяет наличие основных файлов проекта."""
    return (
            path.is_dir()
            and (path / "run.py").is_file()
            and (path / "pyproject.toml").is_file()
    )


def find_project_roots(root: Path) -> list[Path]:
    """Ищет каталоги с run.py и pyproject.toml."""
    roots: set[Path] = set()

    if not root.exists():
        return []

    for pyproject in root.rglob("pyproject.toml"):
        candidate = pyproject.parent

        if is_project_root(candidate):
            roots.add(candidate.resolve())

    return sorted(
        roots,
        key=lambda path: (len(path.parts), str(path)),
    )


def safe_extract_zip(
        archive_path: Path,
        destination: Path,
) -> None:
    """Безопасно распаковывает ZIP в указанный каталог."""
    destination = destination.resolve()

    with zipfile.ZipFile(archive_path) as archive:
        for member in archive.infolist():
            member_path = (
                    destination / member.filename
            ).resolve()

            if (
                    member_path != destination
                    and destination not in member_path.parents
            ):
                raise RuntimeError(
                    f"Небезопасный путь в ZIP: {member.filename}"
                )

        archive.extractall(destination)


def choose_project_root(
        candidates: list[Path],
) -> Path:
    """Выбирает корень проекта из найденных вариантов."""
    if not candidates:
        raise FileNotFoundError(
            "Не найден каталог с run.py и pyproject.toml"
        )

    if len(candidates) == 1:
        return candidates[0]

    preferred = [
        path
        for path in candidates
        if (
                "fl-attacks" in path.name.lower()
                or "fl_attacks" in path.name.lower()
        )
    ]

    if len(preferred) == 1:
        return preferred[0]

    print("Найдено несколько возможных проектов:")

    for index, candidate in enumerate(candidates):
        print(f"  [{index}] {candidate}")

    raise RuntimeError(
        "Невозможно автоматически выбрать проект. "
        "Укажите MANUAL_PROJECT_SOURCE в начале ячейки."
    )


print_input_contents(INPUT_ROOT)

source_path: Path | None = None

if MANUAL_PROJECT_SOURCE is not None:
    source_path = Path(MANUAL_PROJECT_SOURCE).expanduser()

    if not source_path.exists():
        raise FileNotFoundError(
            f"MANUAL_PROJECT_SOURCE не существует: {source_path}"
        )

if source_path is None:
    direct_projects = find_project_roots(INPUT_ROOT)

    if direct_projects:
        source_path = choose_project_root(direct_projects)

if source_path is None:
    zip_files = sorted(
        {
            *INPUT_ROOT.rglob("*.zip"),
            *WORKING_ROOT.glob("*.zip"),
        }
    )

    if len(zip_files) == 1:
        source_path = zip_files[0]
    elif len(zip_files) > 1:
        print("\nНайдено несколько ZIP-файлов:")

        for index, archive_path in enumerate(zip_files):
            print(f"  [{index}] {archive_path}")

        raise RuntimeError(
            "Укажите нужный ZIP в MANUAL_PROJECT_SOURCE."
        )

if source_path is None:
    raise FileNotFoundError(
        "\nПроект не подключён к Kaggle Notebook.\n"
        "Откройте справа Input → Add Input и добавьте Dataset "
        "с ZIP проекта, затем снова выполните эту ячейку."
    )

print("\nВыбранный источник:", source_path)

if source_path.is_file():
    if not zipfile.is_zipfile(source_path):
        raise ValueError(
            f"Источник не является ZIP: {source_path}"
        )

    shutil.rmtree(STAGING_ROOT, ignore_errors=True)
    STAGING_ROOT.mkdir(parents=True, exist_ok=True)

    safe_extract_zip(
        archive_path=source_path,
        destination=STAGING_ROOT,
    )

    extracted_projects = find_project_roots(STAGING_ROOT)
    SOURCE_PROJECT_ROOT = choose_project_root(
        extracted_projects
    )
else:
    if is_project_root(source_path):
        SOURCE_PROJECT_ROOT = source_path.resolve()
    else:
        nested_projects = find_project_roots(source_path)
        SOURCE_PROJECT_ROOT = choose_project_root(
            nested_projects
        )

print("Корень исходного проекта:", SOURCE_PROJECT_ROOT)

PERSIST_ROOT = WORKING_ROOT / "fl-attacks-persistent"
PERSIST_ROOT.mkdir(parents=True, exist_ok=True)

for directory_name in ("data", "outputs"):
    current_directory = PROJECT_ROOT / directory_name
    saved_directory = PERSIST_ROOT / directory_name

    if current_directory.exists():
        shutil.rmtree(saved_directory, ignore_errors=True)
        shutil.move(
            str(current_directory),
            str(saved_directory),
        )
        print(f"Сохранена папка {directory_name}: {saved_directory}")

shutil.rmtree(PROJECT_ROOT, ignore_errors=True)

ignored = shutil.ignore_patterns(
    "../.venv",
    ".git",
    ".idea",
    ".flwr",
    ".flower-local",
    "__pycache__",
    "*.pyc",
    "*.pyo",
    ".DS_Store",
    "data",
    "outputs",
)

shutil.copytree(
    SOURCE_PROJECT_ROOT,
    PROJECT_ROOT,
    ignore=ignored,
)

for directory_name in ("data", "outputs"):
    saved_directory = PERSIST_ROOT / directory_name
    target_directory = PROJECT_ROOT / directory_name

    if saved_directory.exists():
        shutil.move(
            str(saved_directory),
            str(target_directory),
        )
        print(f"Восстановлена папка {directory_name}: {target_directory}")
    else:
        target_directory.mkdir(parents=True, exist_ok=True)

os.chdir(PROJECT_ROOT)

write_test = PROJECT_ROOT / ".kaggle_write_test"
write_test.write_text("ok", encoding="utf-8")
write_test.unlink()

ENV_ROOT = WORKING_ROOT / "fl-attacks-env"
os.environ["UV_PROJECT_ENVIRONMENT"] = str(ENV_ROOT)

gitignore_path = PROJECT_ROOT / ".gitignore"
existing_gitignore = (
    gitignore_path.read_text(encoding="utf-8")
    if gitignore_path.exists()
    else ""
)

required_ignores = (
    ".venv/",
    ".flwr/",
    ".flower-local/",
    "data/",
    "outputs/",
    "__pycache__/",
    "*.pyc",
)

existing_lines = set(existing_gitignore.splitlines())
missing_ignores = [
    pattern
    for pattern in required_ignores
    if pattern not in existing_lines
]

if missing_ignores:
    separator = (
        ""
        if not existing_gitignore
           or existing_gitignore.endswith("\n")
        else "\n"
    )
    gitignore_path.write_text(
        existing_gitignore
        + separator
        + "\n".join(missing_ignores)
        + "\n",
        encoding="utf-8",
    )

print("\nРабочая копия:", PROJECT_ROOT)
print("Внешнее окружение:", ENV_ROOT)
print("Текущая папка:", Path.cwd())
print("Запись в каталог проекта: OK")

Содержимое /kaggle/input:
  datasets/
  datasets/lebibop/
  datasets/lebibop/data-v1-2/
  datasets/lebibop/data-v1-2/fl-attacks-lab/

Выбранный источник: /kaggle/input/datasets/lebibop/data-v1-2/fl-attacks-lab
Корень исходного проекта: /kaggle/input/datasets/lebibop/data-v1-2/fl-attacks-lab

Рабочая копия: /kaggle/working/fl-attacks-lab
Внешнее окружение: /kaggle/working/fl-attacks-env
Текущая папка: /kaggle/working/fl-attacks-lab
Запись в каталог проекта: OK


## 3. Проверка структуры проекта

In [3]:
required_paths = [
    Path("../run.py"),
    Path("../pyproject.toml"),
    Path("../configs/experiments.yaml"),
    Path("../src/fl_attacks"),
]

missing = [
    str(path)
    for path in required_paths
    if not path.exists()
]

if missing:
    raise FileNotFoundError(
        "В проекте отсутствуют обязательные файлы: "
        + ", ".join(missing)
    )

print("Обязательные файлы найдены.\n")
print("Структура проекта:")

for path in sorted(Path("..").glob("*")):
    suffix = "/" if path.is_dir() else ""
    print(f"  {path}{suffix}")

Обязательные файлы найдены.

Структура проекта:
  .gitignore
  README.md
  configs/
  data/
  outputs/
  pyproject.toml
  run.py
  src/
  uv.lock


## 4. Установка или обновление `uv`

In [4]:
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--upgrade",
        "uv",
    ],
    check=True,
)

uv_path = shutil.which("uv")

if uv_path is None:
    raise RuntimeError(
        "Команда uv не найдена после установки. "
        "Проверьте, что Internet включён."
    )

UV = uv_path

version = subprocess.run(
    [UV, "--version"],
    check=True,
    text=True,
    capture_output=True,
).stdout.strip()

print("uv executable:", UV)
print("uv version:", version)

Обновляем uv...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.0/27.0 MB 64.5 MB/s eta 0:00:00
uv executable: /usr/local/bin/uv
uv version: uv 0.11.32 (x86_64-unknown-linux-gnu)


## 5. Создание внешнего Linux-окружения и установка зависимостей

In [5]:
os.chdir(PROJECT_ROOT)

shutil.rmtree(PROJECT_ROOT / ".venv", ignore_errors=True)

RECREATE_ENVIRONMENT = False

if RECREATE_ENVIRONMENT:
    shutil.rmtree(ENV_ROOT, ignore_errors=True)
    print("Старое внешнее окружение удалено")
elif ENV_ROOT.exists():
    print("Используется существующее окружение:", ENV_ROOT)

shutil.rmtree(PROJECT_ROOT / ".flower-local", ignore_errors=True)
shutil.rmtree(PROJECT_ROOT / ".flwr", ignore_errors=True)

install_env = os.environ.copy()
install_env["UV_PROJECT_ENVIRONMENT"] = str(ENV_ROOT)
install_env["UV_LINK_MODE"] = "copy"
install_env["UV_CACHE_DIR"] = "/kaggle/working/.cache/uv"

os.environ["UV_PROJECT_ENVIRONMENT"] = str(ENV_ROOT)

sync_command = [UV, "sync"]

print("Каталог проекта:", PROJECT_ROOT)
print("Каталог окружения:", ENV_ROOT)
print("Выполняется:", " ".join(sync_command))

subprocess.run(
    sync_command,
    cwd=PROJECT_ROOT,
    check=True,
    env=install_env,
)

venv_python = ENV_ROOT / "bin/python"
venv_flwr = ENV_ROOT / "bin/flwr"

if not venv_python.exists():
    raise FileNotFoundError(
        f"Интерпретатор окружения не найден: {venv_python}"
    )

if not venv_flwr.exists():
    raise FileNotFoundError(
        f"Команда Flower не найдена: {venv_flwr}"
    )

if (PROJECT_ROOT / ".venv").exists():
    raise RuntimeError("Внутри проекта неожиданно появилась .venv")

print("Python окружения:", venv_python)
print("Flower CLI:", venv_flwr)


def read_torch_status() -> dict[str, object]:
    """Возвращает версии PyTorch и состояние CUDA."""
    status_script = r'''
import importlib.metadata
import json

result = {}

for package in ("torch", "torchvision"):
    try:
        result[package] = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        result[package] = None

try:
    import torch

    result["cuda_available"] = torch.cuda.is_available()
    result["cuda_build"] = torch.version.cuda
    result["gpu"] = (
        torch.cuda.get_device_name(0)
        if torch.cuda.is_available()
        else None
    )
except Exception as error:
    result["cuda_available"] = False
    result["cuda_build"] = None
    result["gpu"] = None
    result["torch_import_error"] = repr(error)

print(json.dumps(result))
'''

    completed = subprocess.run(
        [str(venv_python), "-c", status_script],
        cwd=PROJECT_ROOT,
        check=True,
        text=True,
        capture_output=True,
        env=install_env,
    )

    return json.loads(completed.stdout.strip())


torch_status = read_torch_status()
print("PyTorch после uv sync:")
print(json.dumps(torch_status, indent=2))

if not bool(torch_status.get("cuda_available")):
    print(
        "\nТекущая сборка PyTorch не видит GPU. "
        "Переустанавливаем CUDA-сборку..."
    )

    pip_help = subprocess.run(
        [UV, "pip", "install", "--help"],
        check=True,
        text=True,
        capture_output=True,
        env=install_env,
    ).stdout

    if "--torch-backend" not in pip_help:
        raise RuntimeError(
            "Установленная версия uv не поддерживает "
            "`uv pip install --torch-backend`."
        )

    package_specs: list[str] = []

    for package in ("torch", "torchvision"):
        version = torch_status.get(package)

        if isinstance(version, str) and version:
            normalized_version = version.split("+", maxsplit=1)[0]
            package_specs.append(f"{package}=={normalized_version}")
        else:
            package_specs.append(package)

    torch_command = [
        UV,
        "pip",
        "install",
        "--python",
        str(venv_python),
        "--reinstall",
        *package_specs,
        "--torch-backend",
        "auto",
    ]

    print("Выполняется:", " ".join(torch_command))

    subprocess.run(
        torch_command,
        cwd=PROJECT_ROOT,
        check=True,
        env=install_env,
    )

    torch_status = read_torch_status()
    print("\nPyTorch после переустановки:")
    print(json.dumps(torch_status, indent=2))

if not bool(torch_status.get("cuda_available")):
    raise RuntimeError(
        "PyTorch во внешнем окружении всё ещё не видит GPU."
    )

print("\nCUDA внутри окружения работает.")
print("GPU:", torch_status.get("gpu"))
print("Внутри проекта .venv отсутствует: OK")

Каталог проекта: /kaggle/working/fl-attacks-lab
Каталог окружения: /kaggle/working/fl-attacks-env
Выполняется: /usr/local/bin/uv sync


Using CPython 3.11.15
Creating virtual environment at: /kaggle/working/fl-attacks-env
Resolved 66 packages in 0.89ms
Prepared 65 packages in 47.13s
Installed 65 packages in 10.20s
 + alembic==1.18.5
 + certifi==2026.7.22
 + cffi==2.1.0
 + charset-normalizer==3.4.9
 + click==8.4.2
 + contourpy==1.3.3
 + cryptography==46.0.7
 + cycler==0.12.1
 + filelock==3.32.0
 + fl-attacks-lab==0.1.0 (from file:///kaggle/working/fl-attacks-lab)
 + flwr==1.32.1
 + fonttools==4.63.0
 + fsspec==2026.6.0
 + greenlet==3.5.4
 + grpcio==1.83.0
 + grpcio-health-checking==1.81.1
 + idna==3.18
 + iterators==0.0.2
 + jinja2==3.1.6
 + kiwisolver==1.5.0
 + mako==1.3.12
 + markdown-it-py==4.2.0
 + markupsafe==3.0.3
 + matplotlib==3.8.4
 + mdurl==0.1.2
 + mpmath==1.3.0
 + networkx==3.6.1
 + numpy==1.26.4
 + nvidia-cublas-cu12==12.1.3.1
 + nvidia-cuda-cupti-cu12==12.1.105
 + nvidia-cuda-nvrtc-cu12==12.1.105
 + nvidia-cuda-runtime-cu12==12.1.105
 + nvidia-cudnn-cu12==8.9.2.26
 + nvidia-cufft-cu12==11.0.2.54
 + nvidia-

Python окружения: /kaggle/working/fl-attacks-env/bin/python
Flower CLI: /kaggle/working/fl-attacks-env/bin/flwr
PyTorch после uv sync:
{
  "torch": "2.2.2",
  "torchvision": "0.17.2",
  "cuda_available": true,
  "cuda_build": "12.1",
  "gpu": "Tesla T4"
}

CUDA внутри окружения работает.
GPU: Tesla T4
Внутри проекта .venv отсутствует: OK


## 6. Проверка CUDA, Flower и модели во внешнем окружении

In [6]:
verification_script = textwrap.dedent(
    """
    import shutil
    import sys

    import flwr
    import torch
    import torchvision

    from fl_attacks.models import create_model

    print("Python:", sys.executable)
    print("Flower:", flwr.__version__)
    print("PyTorch:", torch.__version__)
    print("Torchvision:", torchvision.__version__)
    print("CUDA build:", torch.version.cuda)
    print("CUDA available:", torch.cuda.is_available())

    if not torch.cuda.is_available():
        raise RuntimeError("PyTorch не видит GPU")

    device = torch.device("cuda:0")
    print("GPU:", torch.cuda.get_device_name(0))

    model = create_model(
        in_channels=3,
        num_classes=100,
    ).to(device)

    inputs = torch.randn(
        2,
        3,
        32,
        32,
        device=device,
    )

    with torch.no_grad():
        outputs = model(inputs)

    if tuple(outputs.shape) != (2, 100):
        raise RuntimeError(
            f"Неверная форма выхода: {tuple(outputs.shape)}"
        )

    parameter_count = sum(
        parameter.numel()
        for parameter in model.parameters()
    )

    print("CIFAR-100 output:", tuple(outputs.shape))
    print("Model parameters:", f"{parameter_count:,}")
    print("flwr:", shutil.which("flwr"))
    print("flower-superlink:", shutil.which("flower-superlink"))
    print("flower-supernode:", shutil.which("flower-supernode"))
    """
)

verification_env = os.environ.copy()
verification_env["UV_PROJECT_ENVIRONMENT"] = str(ENV_ROOT)
verification_env["PATH"] = (
        f"{ENV_ROOT / 'bin'}:" + verification_env.get("PATH", "")
)

subprocess.run(
    [str(venv_python), "-c", verification_script],
    cwd=PROJECT_ROOT,
    check=True,
    env=verification_env,
)

Python: /kaggle/working/fl-attacks-env/bin/python
Flower: 1.32.1
PyTorch: 2.2.2+cu121
Torchvision: 0.17.2+cu121
CUDA build: 12.1
CUDA available: True
GPU: Tesla T4
CIFAR-100 output: (2, 100)
Model parameters: 689,668
flwr: /kaggle/working/fl-attacks-env/bin/flwr
flower-superlink: /kaggle/working/fl-attacks-env/bin/flower-superlink
flower-supernode: /kaggle/working/fl-attacks-env/bin/flower-supernode


CompletedProcess(args=['/kaggle/working/fl-attacks-env/bin/python', '-c', '\nimport shutil\nimport sys\n\nimport flwr\nimport torch\nimport torchvision\n\nfrom fl_attacks.models import create_model\n\nprint("Python:", sys.executable)\nprint("Flower:", flwr.__version__)\nprint("PyTorch:", torch.__version__)\nprint("Torchvision:", torchvision.__version__)\nprint("CUDA build:", torch.version.cuda)\nprint("CUDA available:", torch.cuda.is_available())\n\nif not torch.cuda.is_available():\n    raise RuntimeError("PyTorch не видит GPU")\n\ndevice = torch.device("cuda:0")\nprint("GPU:", torch.cuda.get_device_name(0))\n\nmodel = create_model(\n    in_channels=3,\n    num_classes=100,\n).to(device)\n\ninputs = torch.randn(\n    2,\n    3,\n    32,\n    32,\n    device=device,\n)\n\nwith torch.no_grad():\n    outputs = model(inputs)\n\nif tuple(outputs.shape) != (2, 100):\n    raise RuntimeError(\n        f"Неверная форма выхода: {tuple(outputs.shape)}"\n    )\n\nparameter_count = sum(\n    param

## 7. Просмотр конфигурации экспериментов

Ниже выводится текущий `configs/experiments.yaml`.

In [7]:
config_path = Path("../configs/experiments.yaml")

if not config_path.exists():
    raise FileNotFoundError(config_path)

config_text = config_path.read_text(encoding="utf-8")
print(config_text)

# Общие параметры, используемые всеми экспериментами.
defaults:
  # Датасет для всех экспериментов:
  #   mnist
  #   fashionmnist
  #   cifar10
  #   cifar100
  dataset: cifar100

  # Каталог для хранения датасетов.
  data-root: data

  # Доля клиентов, участвующих в обучении в каждом раунде.
  fraction-train: 1.0

  # Доля клиентов, участвующих в оценке модели в каждом раунде.
  fraction-evaluate: 1.0

  # Стратегия агрегации обновлений на сервере:
  #   fedavg        — обычное взвешенное усреднение (базовый вариант);
  #   fedavgm       — FedAvg с серверным momentum;
  #   fedadam       — адаптивная серверная оптимизация Adam;
  #   fedmedian     — покоординатная медиана, устойчивее к выбросам;
  #   fedtrimmedavg — усечённое среднее, отбрасывающее крайние значения.
  strategy: fedavg

  # Параметры FedAvgM и FedAdam.
  # Для FedAvgM используются server-learning-rate и server-momentum.
  # Для FedAdam используются server-learning-rate и параметры fedopt-*.
  server-learning-rate: 1.

## 8. Выбор эксперимента

```python
EXPERIMENT = "clean"
EXPERIMENT = "label_flip"
EXPERIMENT = "sign_flip"
EXPERIMENT = "gaussian_noise"
EXPERIMENT = "backdoor"
EXPERIMENT = "all"
```

In [8]:
EXPERIMENT = "all"

EXTRA_RUN_ARGS: list[str] = []

print("Будет запущен эксперимент:", EXPERIMENT)
print("Дополнительные аргументы:", EXTRA_RUN_ARGS)

Будет запущен эксперимент: all
Дополнительные аргументы: []


## 9. Запуск проекта

In [9]:
def stop_flower_processes() -> None:
    """Завершает локальные процессы Flower, оставшиеся от запуска."""
    patterns = (
        "flower-supernode",
        "flower-superlink",
        "flwr run",
    )

    for pattern in patterns:
        subprocess.run(
            ["pkill", "-f", pattern],
            check=False,
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
        )


os.chdir(PROJECT_ROOT)
stop_flower_processes()

project_venv = PROJECT_ROOT / ".venv"
shutil.rmtree(project_venv, ignore_errors=True)

if project_venv.exists():
    raise RuntimeError("Не удалось удалить .venv внутри проекта")

if not venv_python.exists():
    raise FileNotFoundError(
        f"Внешний Python не найден: {venv_python}"
    )

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
safe_experiment = "".join(
    character
    if character.isalnum() or character in "-_"
    else "_"
    for character in EXPERIMENT
)
LOG_PATH = Path(
    f"/kaggle/working/{safe_experiment}_{timestamp}.log"
)

command = [
    str(venv_python),
    "run.py",
    "--experiment",
    EXPERIMENT,
    *EXTRA_RUN_ARGS,
]

run_env = os.environ.copy()
run_env["PYTHONUNBUFFERED"] = "1"
run_env["MPLBACKEND"] = "Agg"
run_env["CUDA_VISIBLE_DEVICES"] = "0"
run_env["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
run_env["UV_LINK_MODE"] = "copy"
run_env["UV_PROJECT_ENVIRONMENT"] = str(ENV_ROOT)
run_env["PATH"] = f"{ENV_ROOT / 'bin'}:" + run_env.get("PATH", "")

print("Команда:", " ".join(command))
print("Проект:", PROJECT_ROOT)
print("Окружение:", ENV_ROOT)
print("Внутри проекта .venv:", project_venv.exists())
print("Лог:", LOG_PATH)
print("=" * 80)

process = None

try:
    with LOG_PATH.open(
            "w",
            encoding="utf-8",
            buffering=1,
    ) as log_file:
        process = subprocess.Popen(
            command,
            cwd=PROJECT_ROOT,
            env=run_env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )

        assert process.stdout is not None

        for line in process.stdout:
            print(line, end="")
            log_file.write(line)

        return_code = process.wait()

    if return_code != 0:
        raise RuntimeError(
            f"Эксперимент завершился с кодом {return_code}. "
            f"Полный лог: {LOG_PATH}"
        )

    print("=" * 80)
    print("Эксперимент успешно завершён.")
    print("Лог сохранён:", LOG_PATH)

except KeyboardInterrupt:
    if process is not None and process.poll() is None:
        process.terminate()

    print("\nЗапуск остановлен пользователем.")
    raise

finally:
    stop_flower_processes()

Команда: /kaggle/working/fl-attacks-env/bin/python run.py --experiment all
Проект: /kaggle/working/fl-attacks-lab
Окружение: /kaggle/working/fl-attacks-env
Внутри проекта .venv: False
Лог: /kaggle/working/all_20260725_185655.log

=== clean: downloading/checking cifar100 ===

0.0%
0.0%
0.1%
0.1%
0.1%
0.1%
0.1%
0.2%
0.2%
0.2%
0.2%
0.2%
0.3%
0.3%
0.3%
0.3%
0.3%
0.3%
0.4%
0.4%
0.4%
0.4%
0.4%
0.5%
0.5%
0.5%
0.5%
0.5%
0.6%
0.6%
0.6%
0.6%
0.6%
0.7%
0.7%
0.7%
0.7%
0.7%
0.8%
0.8%
0.8%
0.8%
0.8%
0.9%
0.9%
0.9%
0.9%
0.9%
1.0%
1.0%
1.0%
1.0%
1.0%
1.0%
1.1%
1.1%
1.1%
1.1%
1.1%
1.2%
1.2%
1.2%
1.2%
1.2%
1.3%
1.3%
1.3%
1.3%
1.3%
1.4%
1.4%
1.4%
1.4%
1.4%
1.5%
1.5%
1.5%
1.5%
1.5%
1.6%
1.6%
1.6%
1.6%
1.6%
1.6%
1.7%
1.7%
1.7%
1.7%
1.7%
1.8%
1.8%
1.8%
1.8%
1.8%
1.9%
1.9%
1.9%
1.9%
1.9%
2.0%
2.0%
2.0%
2.0%
2.0%
2.1%
2.1%
2.1%
2.1%
2.1%
2.2%
2.2%
2.2%
2.2%
2.2%
2.2%
2.3%
2.3%
2.3%
2.3%
2.3%
2.4%
2.4%
2.4%
2.4%
2.4%
2.5%
2.5%
2.5%
2.5%
2.5%
2.6%
2.6%
2.6%
2.6%
2.6%
2.7%
2.7%
2.7%
2.7%
2.7%
2.8%
2.8%
2.8%
2.8%

## 10. Просмотр созданных результатов

In [10]:
output_root = PROJECT_ROOT / "outputs"

if not output_root.exists():
    raise FileNotFoundError(
        "Папка outputs не найдена. Проверьте лог запуска."
    )

output_files = sorted(
    path
    for path in output_root.rglob("*")
    if path.is_file()
)

if not output_files:
    print("Папка outputs существует, но файлов пока нет.")
else:
    print(f"Найдено файлов: {len(output_files)}")

    for path in output_files:
        print(path.relative_to(PROJECT_ROOT))

Найдено файлов: 40
outputs/2026-07-25__18-57-00__clean/classification_report.json
outputs/2026-07-25__18-57-00__clean/config.yaml
outputs/2026-07-25__18-57-00__clean/confusion_matrix.csv
outputs/2026-07-25__18-57-00__clean/confusion_matrix.png
outputs/2026-07-25__18-57-00__clean/final_model.pt
outputs/2026-07-25__18-57-00__clean/metrics.csv
outputs/2026-07-25__18-57-00__clean/metrics.png
outputs/2026-07-25__18-57-00__clean/summary.json
outputs/2026-07-25__20-24-34__label_flip/classification_report.json
outputs/2026-07-25__20-24-34__label_flip/config.yaml
outputs/2026-07-25__20-24-34__label_flip/confusion_matrix.csv
outputs/2026-07-25__20-24-34__label_flip/confusion_matrix.png
outputs/2026-07-25__20-24-34__label_flip/final_model.pt
outputs/2026-07-25__20-24-34__label_flip/metrics.csv
outputs/2026-07-25__20-24-34__label_flip/metrics.png
outputs/2026-07-25__20-24-34__label_flip/summary.json
outputs/2026-07-25__21-06-03__sign_flip/classification_report.json
outputs/2026-07-25__21-06-03__si

## 11. Упаковка результатов для скачивания

Архив создаётся в `/kaggle/working`. После выполнения он появится в панели файлов Kaggle.

In [12]:
bundle_root = Path("/kaggle/working/fl-attacks-result-bundle")
archive_base = Path("/kaggle/working/fl-attacks-results")

shutil.rmtree(bundle_root, ignore_errors=True)
bundle_root.mkdir(parents=True)

if (PROJECT_ROOT / "outputs").exists():
    shutil.copytree(
        PROJECT_ROOT / "outputs",
        bundle_root / "outputs",
    )

for filename in (
        "pyproject.toml",
        "uv.lock",
):
    source = PROJECT_ROOT / filename

    if source.exists():
        shutil.copy2(source, bundle_root / filename)

config_source = PROJECT_ROOT / "configs/experiments.yaml"

if config_source.exists():
    (bundle_root / "configs").mkdir(exist_ok=True)
    shutil.copy2(
        config_source,
        bundle_root / "configs/experiments.yaml",
    )

if "LOG_PATH" in globals() and Path(LOG_PATH).exists():
    (bundle_root / "logs").mkdir(exist_ok=True)
    shutil.copy2(
        LOG_PATH,
        bundle_root / "logs" / Path(LOG_PATH).name,
    )

archive_path = shutil.make_archive(
    str(archive_base),
    "zip",
    root_dir=bundle_root,
)

print("Архив создан:")
print(archive_path)
print("\nСкачайте его из панели файлов Kaggle.")

Архив создан:
/kaggle/working/fl-attacks-results.zip

Скачайте его из панели файлов Kaggle.
